In [29]:
import os
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)


INPUT_DIR = "/kaggle/input/datasets/ababeiraul/poiero"


train_df = pd.read_csv(f"{INPUT_DIR}/train_data.csv")
test_df = pd.read_csv(f"{INPUT_DIR}/test_data.csv")

EVASION_TO_CLARITY = {
    "Explicit": "Clear Reply",
    "Implicit": "Ambivalent",
    "General": "Ambivalent",
    "Partial/half-answer": "Ambivalent",
    "Dodging": "Ambivalent",
    "Deflection": "Ambivalent",
    "Declining to answer": "Clear Non-Reply",
    "Claims ignorance": "Clear Non-Reply",
    "Clarification": "Clear Non-Reply",
}

EVASION_LABELS = list(EVASION_TO_CLARITY.keys())
CLARITY_LABELS = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

label2id = {lbl: i for i, lbl in enumerate(EVASION_LABELS)}
id2label = {i: lbl for i, lbl in enumerate(EVASION_LABELS)}
clarity2id = {lbl: i for i, lbl in enumerate(CLARITY_LABELS)}

def clean_ro(text):
    if pd.isna(text):
        return ""
    return (
        str(text)
        .replace("ţ", "ț").replace("ş", "ș")
        .replace("Ţ", "Ț").replace("Ş", "Ș")
        .strip()
    )


full_df = pd.concat(
    [
        train_df[["datapointID", "url", "question_order", "interview_question"]].assign(is_test=0),
        test_df[["datapointID", "url", "question_order", "interview_question"]].assign(is_test=1),
    ],
    ignore_index=True,
)
full_df = full_df.sort_values(["url", "question_order", "datapointID"]).reset_index(drop=True)

prev_url = full_df["url"].shift(1)
prev_order = full_df["question_order"].shift(1)
prev_q = full_df["interview_question"].shift(1).apply(clean_ro)


is_consecutive = (full_df["url"] == prev_url) & ((full_df["question_order"] - prev_order) == 1)
full_df["prev_question"] = np.where(is_consecutive, prev_q, "")

prev_q_map = dict(zip(full_df["datapointID"], full_df["prev_question"]))
train_df["prev_question"] = train_df["datapointID"].map(prev_q_map)
test_df["prev_question"] = test_df["datapointID"].map(prev_q_map)

print(f"randuri train q ante consecutiva: {(train_df['prev_question'] != '').sum()} / {len(train_df)}")
print(f"randuri test q ante consecutiva: {(test_df['prev_question'] != '').sum()} / {len(test_df)}")

def head_tail_words(text, head_w=210, tail_w=210):
    words = clean_ro(text).split()
    if len(words) <= head_w + tail_w:
        return " ".join(words)
    return " ".join(words[:head_w]) + " [...] " + " ".join(words[-tail_w:])

def shorten_prev_q(text, max_w=45):
    words = clean_ro(text).split()
    if len(words) <= max_w:
        return " ".join(words)
    return " ".join(words[-max_w:])

def format_inputs(df):
    q_inputs, a_inputs = [], []
    for _, row in df.iterrows():
        pres = clean_ro(row["president"])
        ctx = clean_ro(row["context"])

        q_curr = head_tail_words(row["interview_question"], head_w=80, tail_w=60)
        q_prev = shorten_prev_q(row["prev_question"], max_w=35)

        if q_prev:
            q_str = f"[{pres} | {ctx}] Intrebare anterioara: {q_prev} | Intrebare curenta: {q_curr}"
        else:
            q_str = f"[{pres} | {ctx}] Intrebare: {q_curr}"

        a_str = head_tail_words(row["interview_answer"], head_w=170, tail_w=170)
        q_inputs.append(q_str)
        a_inputs.append(a_str)
    return q_inputs, a_inputs

train_df["q_input"], train_df["a_input"] = format_inputs(train_df)
test_df["q_input"], test_df["a_input"] = format_inputs(test_df)

def tokenize_fn(batch):
    return tokenizer(
        batch["q_input"],
        batch["a_input"],
        truncation="longest_first",
        max_length=512,
    )

test_ds = Dataset.from_pandas(test_df[["q_input", "a_input"]]).map(
    tokenize_fn, batched=True, remove_columns=["q_input", "a_input"]
)


Randuri train q ante consecutiva: 1172 / 3278
Rânduri test q ante consecutiva: 95 / 296


Map:   0%|          | 0/296 [00:00<?, ? examples/s]

In [30]:
def probs9_to_probs3(probs9):
    p_clear = probs9[:, [0]].sum(axis=1)                 # Explicit
    p_ambiv = probs9[:, [1, 2, 3, 4, 5]].sum(axis=1)     # Implicit .. Deflection
    p_nonrep = probs9[:, [6, 7, 8]].sum(axis=1)          # Declining .. Clarification
    return np.stack([p_clear, p_ambiv, p_nonrep], axis=1)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs9 = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds9 = np.argmax(probs9, axis=1)
    f1_evasion = f1_score(labels, preds9, average="macro")

    true_clarity = np.array([clarity2id[EVASION_TO_CLARITY[id2label[l]]] for l in labels])
    pred_clarity = np.argmax(probs9_to_probs3(probs9), axis=1)
    f1_clarity = f1_score(true_clarity, pred_clarity, average="macro")

    return {
        "f1_evasion": f1_evasion,
        "f1_clarity": f1_clarity,
        "combined": 0.5 * (f1_evasion + f1_clarity),
    }


In [31]:
y_evasion = train_df["evasion_label"].map(label2id).values
train_df["label"] = y_evasion

In [34]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_robert9 = np.zeros((len(train_df), 9))
test_robert9 = np.zeros((len(test_df), 9))

folds_run = 0

for fold, (tr_idx, val_idx) in enumerate(skf.split(train_df, y_evasion)):
    print(f"FOLD {fold + 1}")

    tr_ds = Dataset.from_pandas(
        train_df.iloc[tr_idx][["q_input", "a_input", "label"]].reset_index(drop=True)
    ).map(tokenize_fn, batched=True, remove_columns=["q_input", "a_input"])

    val_ds = Dataset.from_pandas(
        train_df.iloc[val_idx][["q_input", "a_input", "label"]].reset_index(drop=True)
    ).map(tokenize_fn, batched=True, remove_columns=["q_input", "a_input"])

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=9)

    args = TrainingArguments(
        output_dir=f"./polero_robert_fold_{fold}",
        num_train_epochs=4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=3e-5,
        weight_decay=0.01,
        warmup_steps = 20,
        fp16=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="combined",
        greater_is_better=True,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tr_ds,
        eval_dataset=val_ds,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )

    trainer.train()

    val_logits = trainer.predict(val_ds).predictions
    oof_robert9[val_idx] = torch.softmax(torch.tensor(val_logits), dim=-1).numpy()

    test_logits = trainer.predict(test_ds).predictions
    test_robert9 += torch.softmax(torch.tensor(test_logits), dim=-1).numpy() / 5.0
    folds_run += 1

# save probs
np.save("oof_robert9.npy", oof_robert9)
np.save("test_robert9.npy", test_robert9)


FOLD 1


Map:   0%|          | 0/2622 [00:00<?, ? examples/s]

Map:   0%|          | 0/656 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dumitrescustefan/bert-base-romanian-cased-v1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

Epoch,Training Loss,Validation Loss,F1 Evasion,F1 Clarity,Combined
1,No log,2.593533,0.144825,0.420233,0.282529
2,No log,2.394877,0.260490,0.607975,0.434232
3,No log,2.373965,0.326717,0.620788,0.473752
4,No log,2.426903,0.347186,0.655777,0.501482


... (60 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [35]:
valid_mask = oof_robert9.sum(axis=1) > 0
oof_robert3 = probs9_to_probs3(oof_robert9[valid_mask])

f1_sub1 = f1_score(y_clarity[valid_mask], np.argmax(oof_robert3, axis=1), average="macro")
f1_sub2 = f1_score(y_evasion[valid_mask], np.argmax(oof_robert9[valid_mask], axis=1), average="macro")

f1_mean = (f1_sub1 + f1_sub2) / 2

print(f"subtask 1 f1 clarity : {f1_sub1:.4f}")
print(f"subtask 2 f1 evasion : {f1_sub2:.4f}")
print(f"scor final: {f1_mean:.4f}")

subtask 1 f1 clarity : 0.6392
subtask 2 f1 evasion : 0.3141
scor final: 0.4767


In [36]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

oof_rob = np.load("oof_robert9.npy")
test_rob = np.load("test_robert9.npy")

In [37]:
test_robert3 = probs9_to_probs3(test_robert9)

In [38]:
def get_score(probs9):
    s1 = f1_score(y_clarity, np.argmax(probs9_to_probs3(probs9), axis=1), average="macro")
    s2 = f1_score(y_evasion, np.argmax(probs9, axis=1), average="macro")
    return s1, s2, 0.5 * (s1 + s2)

s1, s2, best_score = get_score(oof_rob)
print(f"no mult -> sub1: {s1:.4f} | sub2: {s2:.4f} | mean: {best_score:.4f}")

no mult -> sub1: 0.6392 | sub2: 0.3141 | mean: 0.4767


In [39]:
weights = np.ones(9)
for _ in range(2):
    for c in range(9):
        for mult in np.linspace(0.5, 2.5, 21):
            temp_w = weights.copy()
            temp_w[c] = mult
            _, _, score = get_score(oof_rob * temp_w)
            if score > best_score:
                best_score, weights = score, temp_w

In [40]:
s1, s2, mean_s = get_score(oof_rob * weights)
print(f"with mult -> sub1: {s1:.4f} | sub2: {s2:.4f} | mean: {mean_s:.4f}")

with mult -> sub1: 0.6483 | sub2: 0.3700 | mean: 0.5091


In [42]:
test_cal9 = test_rob * weights
test_cal3 = probs9_to_probs3(test_cal9)

In [43]:
sub1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": test_df["datapointID"],
    "answer": [CLARITY_LABELS[i] for i in np.argmax(test_cal3, axis=1)],
})
sub2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": test_df["datapointID"],
    "answer": [id2label[i] for i in np.argmax(test_cal9, axis=1)],
})

submission = pd.concat([sub1, sub2], ignore_index=True)
submission.to_csv("submission_polero_calibrated.csv", index=False)
print(len(submission))
print(submission.head(5))

592
   subtaskID  datapointID       answer
0          1           18   Ambivalent
1          1           23  Clear Reply
2          1           63   Ambivalent
3          1           72  Clear Reply
4          1           89  Clear Reply
